**Production-Grade Code**-
LLM Fine-Tuning Pipeline (QLoRA + SFT + DPO)
This notebook implements an end-to-end, production-ready fine-tuning pipeline tailored for specialized, domain-specific code generation. The entire workflow is optimized to run smoothly within the memory constraints of a free Google Colab NVIDIA T4 GPU (16GB VRAM).

What is this pipeline and how does it work?
Instead of training a model from scratch—which requires massive compute and millions of dollars—this pipeline takes a strong open-source base model and adapts it to a specific coding standard using a two-stage training strategy:

Supervised Fine-Tuning (SFT): Teaches the model how to solve your specific coding tasks and follow exact prompt structures.

Direct Preference Optimization (DPO): Aligns the model by showing it paired examples (clean code vs. buggy/inefficient code), teaching it what to avoid.

End-to-End Workflow
Step 1 — QLoRA Setup: Loads the base model in 4-bit NormalFloat (NF4) precision and attaches trainable Low-Rank Adaptation (LoRA) adapters.

Step 2 — Stage 1 (SFT): Trains the LoRA adapter on instruction-output code pairs using SFTTrainer.

Step 3 — Stage 1 Adapter Merge: Merges the trained SFT adapter directly into the base weights to establish the reference model for Stage 2.

Step 4 — Stage 2 (DPO): Trains a fresh LoRA adapter on pairwise preference data (chosen vs. rejected) using DPOTrainer.

Step 5 — Final Adapter Merge: Consolidates all weights into a standalone, deployable checkpoint.

Step 6 — Qualitative & Execution Evaluation: Validates code correctness with generation prompts and standard execution benchmarks.

Step 7 — Deployment / Hub Upload: Exports and uploads the final weights for serving.

Why this approach? (Key Advantages)
Cost & Hardware Efficiency: QLoRA reduces the required VRAM by ~75%, allowing full fine-tuning on a single 16GB GPU without sacrificing model quality.

High Output Quality: Combining SFT with DPO ensures the model not only produces syntactically valid code, but also prioritizes optimal time complexity, proper error handling, and clean code style.

Production-Ready Artifacts: Merging adapters back into the base model eliminates PEFT overhead during inference, making the output ready for high-throughput serving engines like vLLM, TGI, and SGLang.

Hardware Configuration
Colab Setup: Go to Runtime → Change runtime type → T4 GPU (the free tier provides sufficient memory for the 1.5B parameter model used here). For 7B or 14B models, select an A100/H100 GPU.

Model Selection
Base Model: Qwen/Qwen2.5-Coder-1.5B-Instruct

We use a strong, instruction-tuned coding foundation and continue fine-tuning it on our target domain. This reflects the standard industry practice: adapting a battle-tested instruct model rather than training from raw pre-trained weights.

## 1. Install dependencies (latest stable releases)

In [ ]:

!pip install -q -U transformers accelerate peft trl bitsandbytes datasets huggingface_hub wandb einops

import transformers, peft, trl, accelerate, bitsandbytes, datasets
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("trl         :", trl.__version__)
print("accelerate  :", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("datasets    :", datasets.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires open

## 2. Imports, seed, GPU check

In [ ]:
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTConfig, SFTTrainer, DPOConfig, DPOTrainer
from datasets import load_dataset

assert torch.cuda.is_available(), "GPU nahi mili — Runtime > Change runtime type > T4 GPU select karein"
print("GPU:", torch.cuda.get_device_name(0))


GPU: Tesla T4


## 3. Config

**Global Pipeline Configuration**
This section centralizes all hyperparameter constants, dataset sources, and output directory paths used throughout the fine-tuning and alignment lifecycle.

Why do we configure these variables upfront?
Single Source of Truth: Managing paths and parameters in one central location prevents hardcoding mistakes across different training stages.

Domain Specialization: Narrowing down the task (e.g., Python algorithms, FastAPI endpoints, or data wrangling) allows a compact 1.5B parameter model to achieve domain-specific performance comparable to much larger general models.

Storage & Checkpoint Management: Clear separation between adapter weights (OUTPUT_DIR_SFT, OUTPUT_DIR_DPO) and merged weights (OUTPUT_DIR_SFT_MERGED, OUTPUT_DIR_FINAL) ensures clean state transitions between SFT, DPO, and export stages.

Variable Breakdown & Explanation
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct": The starting foundation model. Starting with a pre-trained instruction-tuned coding model requires drastically less data and compute to reach production accuracy.

SFT_DATASET = "iamtarun/python_code_instructions_18k_alpaca": The Stage 1 dataset containing instruction-input-output code examples used to teach the model target coding formats and syntax.

DPO_DATASET = "Vezora/Code-Preference-Pairs": The Stage 2 preference dataset containing paired responses (chosen vs. rejected) to align the model against common bugs and inefficient code patterns.

OUTPUT_DIR_SFT = "./sft-adapter": Directory where the lightweight Stage 1 LoRA adapter weights are saved after supervised fine-tuning.

OUTPUT_DIR_SFT_MERGED = "./sft-merged": Directory to store the full base model combined with the Stage 1 adapter, which acts as the reference baseline for DPO.

OUTPUT_DIR_DPO = "./dpo-adapter": Directory where the Stage 2 preference alignment LoRA adapter weights are saved.

OUTPUT_DIR_FINAL = "./final-model": The final, fully merged standalone model ready for evaluation and post-training quantization.

MAX_SEQ_LEN = 1024: Maximum token context length per sample. Capping at 1024 tokens balances memory efficiency on 16GB GPUs with sufficient context for code functions.

HF_HUB_MODEL_ID = None: Target Hugging Face repository ID (set to your-username/repo-name when ready to publish).

In [ ]:
BASE_MODEL      = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SFT_DATASET     = "iamtarun/python_code_instructions_18k_alpaca"
DPO_DATASET     = "Vezora/Code-Preference-Pairs"

OUTPUT_DIR_SFT  = "./sft-adapter"
OUTPUT_DIR_SFT_MERGED = "/content/drive/MyDrive/sft-merged"
OUTPUT_DIR_DPO  = "./dpo-adapter"
OUTPUT_DIR_FINAL = "./final-model"

MAX_SEQ_LEN     = 1024
HF_HUB_MODEL_ID = None



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 4. Load base model in 4-bit (QLoRA) + tokenizer

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # NF4 = best accuracy/memory trade-off for QLoRA
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,         # extra memory saving, ~negligible accuracy cost
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",   # Colab T4 par flash-attn-2 available nahi, sdpa best fallback hai
)
model.config.use_cache = False   # gradient checkpointing ke saath zaroori
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 5. LoRA config — attention *aur* MLP layers dono target karein (better result)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",     # attention
        "gate_proj", "up_proj", "down_proj",        # MLP — inhe skip karna common mistake hai
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 6. Stage 1 dataset — SFT (instruction fine-tuning)

In [ ]:
raw_sft = load_dataset(SFT_DATASET, split="train")

#raw_sft = raw_sft.shuffle(seed=SEED).select(range(min(5000, len(raw_sft))))
#if u will train model on whole dataset , training will be run for approx 2.50 hours , so you can select no. of data using range with shuffle
import hashlib
seen = set()
def is_unique(example):
    key = hashlib.md5(example["instruction"].strip().lower().encode()).hexdigest()
    if key in seen:
        return False
    seen.add(key)
    return True

raw_sft = raw_sft.filter(is_unique)

def to_messages(example):
    user_content = example["instruction"]
    if example.get("input"):
        user_content += f"\n\nInput:\n{example['input']}"
    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["output"]},
        ]
    }

sft_dataset = raw_sft.map(to_messages, remove_columns=raw_sft.column_names)


sft_dataset = sft_dataset.train_test_split(test_size=0.05, seed=SEED)
print(sft_dataset)
print(sft_dataset["train"][0])


## 7. Stage 1 training — `SFTConfig` + `SFTTrainer`

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR_SFT,
    max_length=MAX_SEQ_LEN,
    packing=True,

    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,

    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=False,

    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=10,
    report_to="none",
    seed=SEED,
    assistant_only_loss=True,
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"],
    processing_class=tokenizer,
)

sft_trainer.train()
sft_trainer.save_model(OUTPUT_DIR_SFT)
tokenizer.save_pretrained(OUTPUT_DIR_SFT)


Tokenizing train dataset:   0%|          | 0/4745 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4745 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/4745 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,0.593991,0.532090,0.546731,809309.000000,0.849347


('./sft-adapter/tokenizer_config.json',
 './sft-adapter/chat_template.jinja',
 './sft-adapter/tokenizer.json')

In [ ]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.4 MB/s eta 0:00:00


## 8. Merge Stage-1 LoRA adapter into base weights

In [ ]:
#del model, sft_trainer
#torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)
sft_merged = PeftModel.from_pretrained(base_model, OUTPUT_DIR_SFT)
sft_merged = sft_merged.merge_and_unload()

sft_merged.save_pretrained(OUTPUT_DIR_SFT_MERGED)
tokenizer.save_pretrained(OUTPUT_DIR_SFT_MERGED)
print("SFT merged model saved to", OUTPUT_DIR_SFT_MERGED)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SFT merged model saved to ./sft-merged


## 9. Stage 2 dataset — DPO (preference alignment)

Direct Preference Optimization (DPO) aligns the model by training on pairwise comparisons: a prompt, a chosen response (clean, efficient, idiomatic code), and a rejected response (buggy, inefficient, or poorly formatted code).

Why DPO over SFT alone?
Beyond Correctness: SFT teaches the model how to generate code, but DPO explicitly teaches it what anti-patterns to avoid (e.g., infinite loops, unhandled edge cases, suboptimal time complexity).

Simpler than RLHF: Eliminates the complexity of training a separate reward model or tuning PPO stability parameters.

Data Formatting Breakdown
preprocess_dpo: Restructures the dataset columns into standard DPO format:

prompt: The coding question/task provided by the user.

chosen: The high-quality reference solution.

rejected: The flawed or suboptimal code snippet.

train_test_split(test_size=0.05): Reserves 5% of preference pairs to monitor validation loss and reward margins during alignment.

In [ ]:
raw_dpo = load_dataset(DPO_DATASET)
#raw_dpo = raw_dpo.shuffle(seed=SEED).select(range(min(500, len(raw_dpo))))   #same process like sft dataset
def preprocess_dpo(example):
    return {
        "prompt":   [{"role": "user", "content": example["input"]}],
        "chosen":   [{"role": "assistant", "content": example["accepted"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}],
    }

dpo_dataset = raw_dpo.map(preprocess_dpo, remove_columns=raw_dpo["train"].column_names)
dpo_dataset = dpo_dataset["train"].train_test_split(test_size=0.05, seed=SEED)
print(dpo_dataset)


README.md:   0%|          | 0.00/3.34k [00:00<?, ?B/s]

Code-Preference-Pairs.jsonl: reconstructing file:   0%|          |  0.00B /  214MB            

Code-Preference-Pairs.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/54024 [00:00<?, ? examples/s]

Map:   0%|          | 0/54024 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['rejected', 'prompt', 'chosen'],
        num_rows: 51322
    })
    test: Dataset({
        features: ['rejected', 'prompt', 'chosen'],
        num_rows: 2702
    })
})


## 10. Re-apply fresh LoRA adapter on merged SFT model, then Stage 2 — `DPOConfig` + `DPOTrainer`

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

policy_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR_SFT_MERGED,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
policy_model.config.use_cache = False
policy_model = prepare_model_for_kbit_training(policy_model, use_gradient_checkpointing=True)

dpo_lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
policy_model = get_peft_model(policy_model, dpo_lora_config)

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR_DPO,
    beta=0.1,                              # DPO temperature —

    num_train_epochs=1,                    # DPO usually need 1 epochs , on giving more model , it could be collapse
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-6,                    # use lower learning rate , DPO is sensative as compare to SFT
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,

    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,

    max_length=512,

    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,

    logging_steps=5,
    report_to="none",
    seed=SEED,
)

dpo_trainer = DPOTrainer(
    model=policy_model,
    args=dpo_config,
    train_dataset=dpo_dataset["train"],
    eval_dataset=dpo_dataset["test"],
    processing_class=tokenizer,
    # reference model pass nahi kiya — PEFT use karne par TRL automatically
    # adapter disable karke base weights ko hi reference ki tarah use karta hai (memory-efficient)
)

dpo_trainer.train()
dpo_trainer.save_model(OUTPUT_DIR_DPO)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.655232,0.642413,0.573132,358695.000000,-3.162111,-2.943056,0.848403,0.032626,-0.072823,1.000000,0.105449,-132.160095,-170.519253


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.655232,0.642413,0.573132,358695.000000,-3.162111,-2.943056,0.848403,0.032626,-0.072823,1.000000,0.105449,-132.160095,-170.519253
58,0.650471,0.636524,0.573132,409724.000000,-3.161696,-2.942654,0.849213,0.034505,-0.083653,1.000000,0.118158,-132.141307,-170.627557


## 11. Merge final DPO adapter → deployable model

In [ ]:
del policy_model, dpo_trainer
torch.cuda.empty_cache()

base_for_final = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR_SFT_MERGED, torch_dtype=torch.bfloat16, device_map="auto"
)
final_model = PeftModel.from_pretrained(base_for_final, OUTPUT_DIR_DPO)
final_model = final_model.merge_and_unload()

final_model.save_pretrained(OUTPUT_DIR_FINAL)
tokenizer.save_pretrained(OUTPUT_DIR_FINAL)
print("Final deployable model saved to", OUTPUT_DIR_FINAL)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final deployable model saved to ./final-model


## 12. Evaluation

Validation loss was tracked across both Stage 1 and Stage 2 training runs (with the optimal checkpoint restored via load_best_model_at_end=True). Here, we perform a manual inspection to verify output structure and syntax.

Why we run qualitative checks:

Ensures the model adheres strictly to the chat template.

Verifies that code output is complete, free of syntax errors, and well-structured.

Code Breakdown:

final_model.eval(): Sets the model to evaluation mode (disables dropout).

tokenizer.apply_chat_template(...): Formats user prompts into the exact special-token structure expected by Qwen.

`torch.no_grad()`: Disables gradient tracking to save VRAM and speed up generation.

final_model.generate(...): Generates code using low temperature (0.2) for deterministic, focused logic.

In [ ]:
test_prompts = [
    "Write a Python function that returns the nth Fibonacci number using memoization.",
    "Write a function to check if a string is a palindrome, ignoring case and spaces.",
]

final_model.eval()
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]


    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False
    ).to(final_model.device)

    with torch.no_grad():
        out = final_model.generate(
            input_ids,
            max_new_tokens=300,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    print("PROMPT:", prompt)
    print(tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True))
    print("-" * 80)

PROMPT: Write a Python function that returns the nth Fibonacci number using memoization.
def fibonacci(n):
    # Base case: 0th and 1st Fibonacci numbers
    if n == 0:
        return 0
    elif n == 1:
        return 1
    
    # Memoization dictionary to store previously computed values
    memo = {}
    
    # Recursive helper function with memoization
    def fib_helper(n):
        if n in memo:
            return memo[n]
        
        # Compute the Fibonacci number recursively
        result = fib_helper(n-1) + fib_helper(n-2)
        memo[n] = result
        
        return result
    
    # Call the recursive helper function
    return fib_helper(n)
--------------------------------------------------------------------------------
PROMPT: Write a function to check if a string is a palindrome, ignoring case and spaces.
def is_palindrome(s):
    s = s.lower().replace(" ", "")
    return s == s[::-1]
--------------------------------------------------------------------------------


**Quantitative benchmark chahiye ho to** (recommended for real production
validation): `bigcode-evaluation-harness` use karke HumanEval / MBPP par
`pass@1` score nikalein, base model vs aapke fine-tuned model ka compare
karke improvement confirm karein — ye separate repo hai isliye is notebook
ke scope se bahar rakha hai, but agli baar poochh sakte hain, main uska
setup bhi bana dunga.


## 13. Standardized benchmark evaluation — `lm-evaluation-harness`

(lm-evaluation-harness)
Qualitative checks and loss metrics only provide sanity checks. Production LLM validation requires standard, reproducible execution benchmarks like HumanEval and MBPP.

Why standard benchmarks matter:

Real Execution (pass@1): Tests functional correctness by executing generated code against isolated unit tests rather than just checking text similarity.

Regression Detection: Comparing baseline scores (BASE_MODEL) against the fine-tuned model confirms real improvement and flags potential degradation from overfitting or bad preference data.

Parameter & Flag Breakdown:

--tasks humaneval,mbpp: Runs standard Python algorithmic evaluation suites.

--confirm_run_unsafe_code: Allows execution of model-generated code inside the sandboxed Colab runtime.

--output_path ./benchmark_results: Exports test scores and execution logs.


In [ ]:
!pip install -q lm-eval[api]

!lm_eval \
  --model hf \
  --model_args pretrained={OUTPUT_DIR_FINAL},dtype=bfloat16 \
  --tasks humaneval,mbpp \
  --batch_size 8 \
  --confirm_run_unsafe_code \
  --output_path ./benchmark_results


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 8.4 MB/s eta 0:00:00
2026-08-18:14:58:45 INFO     [_cli.run:388] Selected Tasks: ['humaneval', 'mbpp']
2026-08-18:14:58:49 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-08-18:14:58:49 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': './final-model', 'dtype': 'bfloat16'}
2026-08-18:

## 14. Post-training quantization for deployment (AWQ)

Training ke liye QLoRA (4-bit, on-the-fly) use kiya — lekin wo **inference**
ke liye optimize nahi hai. Deployment ke liye alag se ek **calibrated**
quantization karte hain jisse:
- Model size ~4x chhota ho jaata hai (disk + VRAM)
- Inference throughput badh jaata hai (kam memory bandwidth chahiye)
- Accuracy loss negligible rehta hai (AWQ calibration data use karke important weights ko protect karta hai)

`AWQ` (Activation-aware Weight Quantization) is samay ka industry-standard
hai for deployment — vLLM, TGI, aur SGLang sab natively support karte hain.


In [ ]:
!pip install -q autoawq

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

QUANT_DIR = "./final-model-awq"

awq_model = AutoAWQForCausalLM.from_pretrained(OUTPUT_DIR_FINAL, safetensors=True)
awq_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR_FINAL)

quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}


awq_model.quantize(awq_tokenizer, quant_config=quant_config)

awq_model.save_quantized(QUANT_DIR)
awq_tokenizer.save_pretrained(QUANT_DIR)
print(f"Deployment-ready 4-bit AWQ model saved to {QUANT_DIR}")
print("Is model ko vLLM / TGI / SGLang mein directly load kiya ja sakta hai fast serving ke liye.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/167 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


...

val.jsonl.zst: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/214670 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (57053 > 32768). Running this sequence through the model will result in indexing errors
AWQ: 100%|██████████| 28/28 [25:44<00:00, 55.16s/it]


Writing model shards: 0it [00:00, ?it/s]

Deployment-ready 4-bit AWQ model saved to ./final-model-awq
Is model ko vLLM / TGI / SGLang mein directly load kiya ja sakta hai fast serving ke liye.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

HF_HUB_MODEL_ID = "Anshrajsingh/qwen2.5-coder-1.5b-awq"

api = HfApi()
api.create_repo(repo_id=HF_HUB_MODEL_ID, exist_ok=True, private=False)

api.upload_folder(
    folder_path="./final-model-awq",
    repo_id=HF_HUB_MODEL_ID,
    repo_type="model",
    token=hf_token
)

print(f"🚀 Model successfully uploaded to: https://huggingface.co/{HF_HUB_MODEL_ID}")

🚀 Model successfully uploaded to: https://huggingface.co/Anshrajsingh/qwen2.5-coder-1.5b-awq


## 17. (Optional) Push final model to Hugging Face Hub

> Add blockquote



In [ ]:
# from huggingface_hub import login
# login()
#
# if HF_HUB_MODEL_ID:
#     final_model.push_to_hub(HF_HUB_MODEL_ID)
#     tokenizer.push_to_hub(HF_HUB_MODEL_ID)
#     print(f"Pushed to https://huggingface.co/{HF_HUB_MODEL_ID}")
